In [ ]:
!pip install -qU langgraph langchain langchain-google-genai langchain-openai

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-xxx"
os.environ["GEMINI_API_KEY"] = "AIzaxxxx"

[Workflow 공식문서](https://docs.langchain.com/oss/python/langgraph/workflows-agents#orchestrator-worker)

# Prompt Chaining

## Model 정의

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano", temperature=1.0)

## State 정의

In [ ]:
from typing import TypedDict

class JokeState(TypedDict):
    topic: str          # 초기 주제
    draft_joke: str     # 1단계: 초안
    improved_joke: str  # 2단계: 수정안
    final_joke: str     # 3단계: 최종 결과

## Node 정의

In [ ]:
# [Step 1] 초안 생성 노드
def generate_joke(state: JokeState):
    topic = state["topic"]
    print(f"\n--- [1단계] '{topic}' 주제로 농담 초안 생성 중 ---")

    msg = model.invoke(f"'{topic}'에 대한 짧고 재미있는 농담을 하나 만들어줘. 다른 말은 하지 말고 농담 하나만 해")
    return {"draft_joke": msg.content}

In [ ]:
# [Step 2] 윤색/수정 노드
def critique_and_improve(state: JokeState):
    original_joke = state["draft_joke"]
    print(f"\n--- [2단계] 더 웃기게 수정 중 (아재개그 스타일) ---")

    prompt = f"""
    다음 농담을 보고, 더 썰렁하고 재미있는 '아재개그' 스타일로 개선해줘. 다른 말은 하지 말고 하나만 답변해.
    원문: {original_joke}
    """
    msg = model.invoke(prompt)
    return {"improved_joke": msg.content}

In [ ]:
# [Step 3] 최종 포장 노드
def polish_joke(state: JokeState):
    improved_joke = state["improved_joke"]
    print(f"\n--- [3단계] 이모지 추가 및 마무리 ---")

    prompt = f"""
    다음 농담에 적절한 이모지를 듬뿍 넣어서 SNS에 올리기 좋게 꾸며줘.
    농담: {improved_joke}
    """
    msg = model.invoke(prompt)
    return {"final_joke": msg.content}

## 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 객체 생성

# 노드 등록


# 엣지 연결: 일직선으로 연결 (Linear Chain)
# START -> 1단계 -> 2단계 -> 3단계 -> END


# 컴파일


In [ ]:
chain

## 실행

In [ ]:
inputs = {"topic": "고양이"}
result = chain.invoke(inputs)

In [ ]:
print(f"주제: {result['topic']}")
print(f"1차 초안: {result['draft_joke']}")
print(f"2차 수정: {result['improved_joke']}")
print("-" * 30)
print(f"최종 완성:\n{result['final_joke']}")

# Parallelization (병럴 처리)

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano")

## State 정의

In [ ]:
class WriterState(TypedDict):
    topic: str          # 주제
    poem: str           # 시 (결과물 A)
    story: str          # 소설 (결과물 B)
    joke: str           # 농담 (결과물 C)
    final_report: str   # 최종 편집본

## Node 정의

In [ ]:
# [Worker A] 시인 노드
def write_poem(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker A] '{topic}' 주제로 시(Poem) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 아름다운 시를 짧게 써줘.")
    return {"poem": msg.content}

In [ ]:
# [Worker B] 소설가 노드
def write_story(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker B] '{topic}' 주제로 소설(Story) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 감동적인 짧은 이야기를 써줘.")
    return {"story": msg.content}

In [ ]:
# [Worker C] 개그맨 노드
def write_joke(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker C] '{topic}' 주제로 농담(Joke) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 재미있는 아재개그를 하나 해줘.")
    return {"joke": msg.content}

In [ ]:
# [Aggregator] 편집장 노드 (모든 결과를 취합)
def aggregator(state: WriterState):
    print("\n--- [Aggregator] 모든 원고 도착! 최종 편집 중 ---")

    # State에 저장된 각 결과물을 꺼내서 합침
    final_text = f"""
    [주제: {state['topic']} 종합 선물세트]

    1. 시 (Poem)
    ----------------
    {state['poem']}

    2. 소설 (Story)
    ----------------
    {state['story']}

    3. 농담 (Joke)
    ----------------
    {state['joke']}
    """
    return {"final_report": final_text}

## 그래프 생성

In [ ]:
workflow = StateGraph(WriterState)

# 노드 추가


# 엣지 연결: START에서 3개의 노드로 동시에 뻗어 나가기 (Fan-out)


# 3개의 노드가 모두 하나의 노드로 모이기 (Fan-in)


# 컴파일


In [ ]:
app

## 실행

In [ ]:
inputs = {"topic": "직장인의 월요일"}
result = app.invoke(inputs)

In [ ]:
result

In [ ]:
result["final_report"]

# Routing

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano")

## State 정의

In [ ]:
class SupportState(TypedDict):
    query: str          # 고객 질문
    category: str       # 라우터가 분류한 카테고리
    response: str       # 전문가의 답변

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):


In [ ]:
router_llm

## Node 정의

In [ ]:
# [Manager] router 노드
def router_node(state: SupportState):
    query = state["query"]
    print(f"\n--- [Router] 문의 분석 중: '{query}' ---")

    # LLM이 문맥을 읽고 카테고리를 결정함 (Rule-based가 아님!)


    print(f"   -> 분류 결과: {decision.category.upper()} 부서로 연결합니다.")
    return {"category": decision.category}

In [ ]:
# [Expert A] 결제/환불 담당
def billing_expert(state: SupportState):
    print("--- [Billing Expert] 결제 전문가가 답변 작성 중 ---")
    prompt = f"당신은 결제 및 환불 전문가입니다. 다음 문의에 대해 정중하게 답변하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert B] 기술 지원 담당
def technical_expert(state: SupportState):
    print("--- [Tech Expert] 기술 지원 엔지니어가 분석 중 ---")
    prompt = f"당신은 IT 엔지니어입니다. 다음 기술 문제에 대해 해결책을 제시하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert C] 배송 담당
def shipping_expert(state: SupportState):
    print("--- [Shipping Expert] 물류 담당자가 배송 조회 중 ---")
    prompt = f"당신은 배송 관리자입니다. 다음 배송 문의에 대해 답변하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert D] 일반 상담
def general_expert(state: SupportState):
    print("--- [General] 일반 상담원이 답변 중 ---")
    msg = model.invoke(f"다음 문의에 친절하게 답변하세요: {state['query']}")
    return {"response": msg.content}

## 그래프 생성

In [ ]:
workflow = StateGraph(SupportState)

# 노드 등록
workflow.add_node("router_node", router_node)
workflow.add_node("billing_expert", billing_expert)
workflow.add_node("technical_expert", technical_expert)
workflow.add_node("shipping_expert", shipping_expert)
workflow.add_node("general_expert", general_expert)

# 엣지 연결
workflow.add_edge(START, "router_node")

In [ ]:
# 조건부 엣지를 위한 함수
def route_to_expert(state: SupportState):


In [ ]:
workflow.add_conditional_edges(

)

In [ ]:
app = workflow.compile()

In [ ]:
app

## 실행

In [ ]:
app.invoke({"query": "지난달 요금이 두 번 빠져나갔어요. 확인해주세요."})

In [ ]:
app.invoke({"query": "API 연결할 때 404 에러가 자꾸 떠요."})

In [ ]:
app.invoke({"query": "주문한 노트북 언제 도착하나요?"})

# Evaluator-Optimizer

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano", temperature=0.7)

## State 정의

In [ ]:
class AdState(TypedDict):
    product_name: str       # 상품명
    ad_copy: str            # 생성된 광고 문구



In [ ]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):


In [ ]:
evaluator_llm

## Node 정의

In [ ]:
# [Generator] 신입 카피라이터
def copywriter_node(state: AdState):
    print(f"\n--- [Copywriter] 광고 문구 작성 중 (시도: {count + 1}) ---")

    if not feedback:
        # 첫 시도는 좀 건조하게 작성하도록 유도 (Fail을 유발하기 위해)
        prompt = f"'{product}'의 기능 위주로 인스타그램 홍보 문구를 건조하게 작성해줘. 홍보 문구만 답변하고 반드시 20자 이하로 작성하시오."
    else:
        # 피드백 반영
        prompt = f"""
        '{product}' 인스타그램 홍보 문구를 다시 작성해.

        <반드시 지켜야 할 수정 사항>
        {feedback}
        </반드시 지켜야 할 수정 사항>

        <작성 시 반드시 지켜야할 사항>
        홍보 문구만 답변하고 절대 50자 이하로 작성하시오.
        </작성 시 반드시 지켜야할 사항>
        """


In [ ]:
# [Evaluator] 감성적인 마케팅 팀장
def manager_node(state: AdState):
    print(f"\n--- [Manager] 문구 검수 중 ---")

    print(f"   ㄴ 신입이 쓴 글: {ad_copy}")

    prompt = f"""
    당신은 깐깐한 마케팅 팀장입니다. 신입 사원이 쓴 다음 광고 문구를 평가하세요:

    "{ad_copy}"

    <평가 기준>
    1. (정량) 해시태그(#)가 3개 이상 있어야 합니다.
    2. (정량) '할인' 또는 '특가'라는 단어가 포함되어야 합니다.
    3. (정성 - 중요!) **문구가 너무 설명문 같거나 딱딱하면 안 됩니다. 소비자의 감성을 자극하는 '활기차고 매력적인 톤'이어야 합니다.**
    </평가 기준>

    위 3가지 기준 중 하나라도 부족하면 fail을 주세요.
    특히 3번(톤앤매너)이 부족하다면 "좀 더 감성적으로 쓰세요" 같이 100자 이내로 조언하세요.
    """


## 그래프 생성

In [ ]:
# 4. 루프 로직
def route_submission(state: AdState):


In [ ]:
# 5. 그래프 생성
workflow = StateGraph(AdState)
workflow.add_node("copywriter_node", copywriter_node)
workflow.add_node("manager_node", manager_node)


In [ ]:
app = workflow.compile()

In [ ]:
app

## 실행

In [ ]:
inputs = {"product_name": "자율주행 자동차"}
result = app.invoke(inputs)

# Orchestrator-Worker

## Model 정의

In [ ]:
model = init_chat_model("gpt-5-nano")

## State 정의

In [ ]:
from typing import List

# 1. 데이터 모델 정의 (Plan & Section)
# 팀장(Orchestrator)이 짤 계획표의 양식
class Section(BaseModel):


In [ ]:
import operator
from typing import Annotated

# 2. State 정의 (Reducer 필수!)
class ReportState(TypedDict):


In [ ]:
# Worker에게 전달할 별도의 State (작업 지시서)
class WorkerState(TypedDict):
    section: Section

In [ ]:
class Plan(BaseModel):
    sections: List[Section] = Field(description="보고서 작성을 위한 목차 리스트")

In [ ]:
planner_llm = model.with_structured_output(Plan)

## Node 정의

In [ ]:
# [Orchestrator] 팀장 노드: 계획 수립
def orchestrator_node(state: ReportState):
    topic = state["topic"]
    print(f"\n--- [Orchestrator] '{topic}' 보고서 계획 수립 중 ---")

    # 보고서 목차를 생성 (최대 3개로 제한하여 속도 조절)
    plan = planner_llm.invoke(f"'{topic}'에 대한 보고서 목차를 짜줘. 3개 섹션 이내로 구성해.")

    print(f"생성된 계획: {[s.name for s in plan.sections]}")
    return {"sections": plan.sections}

In [ ]:
# [Worker] 팀원 노드: 섹션 집필
# 이 노드는 여러 개가 복제되어 동시에 실행
def worker_node(state: WorkerState):
    section = state["section"]
    print(f"   --- [Worker] 집필 중: {section.name} ---")

    prompt = f"""
    다음 섹션에 대한 내용을 짧게 작성해줘.
    제목: {section.name}
    내용 가이드: {section.description}
    """


In [ ]:
# [Synthesizer] 편집자 노드: 취합
def synthesizer_node(state: ReportState):
    print("\n--- [Synthesizer] 모든 원고 취합 및 최종 편집 ---")

    # 리스트에 모인 조각글들을 하나로 합침
    completed = state["completed_sections"]
    final_report = "\n".join(completed)

    return {"final_report": final_report}

## 그래프 생성

### [Send API](https://reference.langchain.com/python/langgraph/types/?h=send#langgraph.types.Send)

In [ ]:
from langgraph.types import Send

# 5. 동적 라우팅 로직 (Map: 1 -> N)
# 일반적인 edge가 아니라, 리스트 개수만큼 노드를 생성해서 뿌려주는 역할
def assign_workers(state: ReportState):


In [ ]:
# 6. 그래프 조립


# 6-1. 시작 -> 팀장


# 6-2. 팀장 -> (Map) -> 팀원들 (Conditional Edge)
workflow.add_conditional_edges(

)

# 6-3. 팀원들 -> (Reduce) -> 편집자


# 6-4. 편집자 -> 종료


In [ ]:
app = workflow.compile()

In [ ]:
app

## 실행

In [ ]:
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
result["final_report"]

### develop (화학적 결합)

In [ ]:
# [Synthesizer] 편집자 노드: 단순 취합이 아니라 '화학적 결합'을 수행
def synthesizer_node(state: ReportState):
    topic = state["topic"]
    completed_docs = state["completed_sections"]

    print(f"\n--- [Synthesizer] 원고 {len(completed_docs)}건 도착. 최종 편집 시작 ---")

    # 1. 일단 텍스트 덩어리로 병합


    # 2. LLM에게 '전문 편집자' 역할 부여
    prompt = f"""
    당신은 전문 리포트 편집자입니다.
    다음은 '{topic}'에 대해 여러 작가가 나누어 쓴 원고들입니다.

    이 초안들을 바탕으로 **하나의 자연스럽고 전문적인 보고서**로 다시 작성해주세요.

    [지시사항]
    1. 각 섹션의 연결이 매끄러워야 합니다.
    2. 전체를 아우르는 '서론'과 '결론'을 추가해주세요.
    3. 마크다운(Markdown) 형식을 사용하여 가독성을 높여주세요.

    [원고 내용]
    {raw_content}
    """

    # 최종 생성을 위한 LLM 호출



In [ ]:
# 6. 그래프 조립
workflow = StateGraph(ReportState)

workflow.add_node("orchestrator_node", orchestrator_node)
workflow.add_node("worker_node", worker_node)
workflow.add_node("synthesizer_node", synthesizer_node)

# 1. 시작 -> 팀장
workflow.add_edge(START, "orchestrator_node")

# 2. 팀장 -> (Map) -> 팀원들 (Conditional Edge)
workflow.add_conditional_edges(
    "orchestrator_node",
    assign_workers,
    ["worker_node"] # 이 부분은 생략 가능하지만 명시적으로 적어줌
)

# 3. 팀원들 -> (Reduce) -> 편집자
workflow.add_edge("worker_node", "synthesizer_node")

# 4. 편집자 -> 종료
workflow.add_edge("synthesizer_node", END)

In [ ]:
app = workflow.compile()

In [ ]:
app

In [ ]:
inputs = {"topic": "생성형 AI의 미래"}
result = app.invoke(inputs)

In [ ]:
result["final_report"]